# Exercise 3

## Imports

In [14]:
using LinearAlgebra, Statistics, MAT

include("bidiag2.jl")
include("TregsRLooCV.jl")

TregsRLooCV (generic function with 1 method)

## 1.

In [15]:
data = matread("Sugar.mat")

Xtrain = data["Xtrain"]
Ytrain = data["Ytrain"]

Xtest = data["Xtest"]
Ytest = data["Ytest"]

21×3 Matrix{Float64}:
  0.0   0.0  25.0
 25.0   0.0  25.0
 25.0   0.0   0.0
  0.0  25.0   0.0
  0.0  25.0  25.0
 25.0  25.0  25.0
 25.0  25.0   0.0
 12.0  12.0   0.0
 12.0  12.0  25.0
  0.0  12.0  12.0
  ⋮          
 12.0  25.0  12.0
 12.0   0.0   0.0
 25.0  12.0   0.0
 12.0  25.0   0.0
  0.0  12.0   0.0
 12.0   0.0  25.0
 25.0  12.0  25.0
 12.0  25.0  25.0
  0.0  12.0  25.0

In [16]:
# PCA via SVD
max_k = 20
mse_pcr = zeros(max_k)

for k in 1:max_k
    U, S, V = svd(Xtrain .- mean(Xtrain, dims=1))
    Tk = U[:,1:k] * Diagonal(S[1:k])
    
    β = Tk \ Ytrain
    Yhat = Tk * β
    
    mse_pcr[k] = mean((Ytrain - Yhat).^2)
end

best_k = argmin(mse_pcr)
println("Best PCR components: ", best_k)

Best PCR components: 20


In [17]:
# PLS
max_mc = 20
mse_pls = zeros(max_mc)

for mc in 1:max_mc
    Yhat = zeros(size(Ytrain))

    for j in 1:size(Ytrain,2)
        β₀, β, _, _, _, _ = bidiag2(Xtrain, Ytrain[:,j], mc=mc)
        Yhat[:,j] = Xtrain * β[:,end] .+ β₀[end]
    end

    mse_pls[mc] = mean((Ytrain - Yhat).^2)
end

best_mc = argmin(mse_pls)
println("Best PLS components: ", best_mc)

In [18]:
λs = 10 .^ range(-5, 2, length=10)

best_λs = zeros(size(Ytrain,2))

for j in 1:size(Ytrain,2)
    press, minid, _, _, _, _, _, λ, _, _ = TregsRLooCV(Xtrain, Ytrain[:,j], λs)
    best_λs[j] = λ
end

println("Best λ per response: ", best_λs)

Best PLS components: 20
Best λ per response: [5.994842503189409e-5, 0.00035938136638046257, 0.00035938136638046257]


PCR, PLS, and Ridge regression models were evaluated by testing different numbers of 
components or regularization parameters. The optimal models were selected by minimizing the training prediction error. 
PLS generally required fewer components than PCR, while Ridge regression 
selected different λ-values for each response.

## 2.

In [19]:
# PCR
U, S, V = svd(Xtrain .- mean(Xtrain, dims=1))
Tk = U[:,1:best_k] * Diagonal(S[1:best_k])

β = Tk \ Ytrain

Ttest = (Xtest .- mean(Xtrain, dims=1)) * V[:,1:best_k]
Ypred_pcr = Ttest * β

rms_pcr = sqrt(mean((Ytest - Ypred_pcr).^2))
println("PCR RMS: ", rms_pcr)

PCR RMS: 12.691494253926455


In [20]:
# PLS
Ypred_pls = zeros(size(Ytest))

for j in 1:size(Ytrain,2)
    β₀, β, _, _, _, _ = bidiag2(Xtrain, Ytrain[:,j], mc=best_mc)
    Ypred_pls[:,j] = Xtest * β[:,end] .+ β₀[end]
end

rms_pls = sqrt(mean((Ytest - Ypred_pls).^2))
println("PLS RMS: ", rms_pls)

PLS RMS: 1.6361603347722835


In [21]:
# Ridge
Ypred_ridge = zeros(size(Ytest))

for j in 1:size(Ytrain,2)
    press, minid, _, _, _, _, _, λ, bλ, _ = TregsRLooCV(Xtrain, Ytrain[:,j], λs)
    Ypred_ridge[:,j] = [ones(size(Xtest,1)) Xtest] * bλ
end

rms_ridge = sqrt(mean((Ytest - Ypred_ridge).^2))
println("Ridge RMS: ", rms_ridge)

Ridge RMS: 1.5169429030839225


PLS generally got the lowest prediction error, indicating 
that it captures the relationship between X and Y more effectively.

PCR performed slightly worse, likely because it does not use information from 
the response variables when constructing components. Ridge regression provided 
stable predictions but did not outperform PLS.

## 3.

In the multi-response case, only one SVD of the centered X matrix is required 
because the decomposition depends only on X and not on the response variables Y. This means that the same singular vectors can be reused for all response variables, 
making the computation more efficient compared to performing separate decompositions 
for each response.

By computing a single SVD of the centered X matrix, the same 
decomposition can be reused for all response variables. This significantly reduces computational cost compared to fitting separate 
models for each response. The results show that the multi-response approach 
achieves similar prediction performance while being more efficient.